# Financial RAG — Evaluation Notebook

Interactive walkthrough of the pipeline: ingest → chunk → index → retrieve → rerank → evaluate.
Run from the project root so that the `src` package is importable.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.config import DATA_DIR, SETTINGS
from src.loaders import load_directory
from src.chunking import chunk_documents, chunk_statistics
from src.vector_store import build_index
print('Embedding model:', SETTINGS.embedding_model)

## 1. Load documents

In [ ]:
docs = load_directory(DATA_DIR)
print(f'Loaded {len(docs)} pages/sections')
set(d.metadata['doc_type'] for d in docs)

## 2. Chunk with both strategies and compare statistics

In [ ]:
import pandas as pd

fixed_chunks = chunk_documents(docs, 'fixed')
semantic_chunks = chunk_documents(docs, 'semantic')

stats = pd.DataFrame({
    'fixed': chunk_statistics(fixed_chunks),
    'semantic': chunk_statistics(semantic_chunks),
})
stats

## 3. Build FAISS indexes

In [ ]:
build_index(fixed_chunks, 'fixed')
build_index(semantic_chunks, 'semantic')
print('Indexes built.')

## 4. Run the four experiments (A–D)

In [ ]:
from src.rag_pipeline import build_pipeline
from src.evaluation import load_eval_questions, evaluate_experiment

questions = load_eval_questions()
EXPERIMENTS = [('A','fixed',False),('B','fixed',True),('C','semantic',False),('D','semantic',True)]

rows = []
for label, method, rerank in EXPERIMENTS:
    pipe = build_pipeline(method=method, use_reranker=rerank)
    answered = [(q, pipe.answer(q.question)) for q in questions]
    m = evaluate_experiment(answered, run_ragas=False)['retrieval']
    rows.append({'exp': label, 'chunking': method, 'rerank': rerank, **m})

retrieval_df = pd.DataFrame(rows)
retrieval_df

## 5. Visualise retrieval metrics

In [ ]:
import matplotlib.pyplot as plt

ax = retrieval_df.set_index('exp')[['precision@k','recall@k','mrr']].plot.bar(figsize=(8,4))
ax.set_title('Retrieval metrics by experiment')
ax.set_ylabel('score')
plt.tight_layout(); plt.show()

## 6. (Optional) RAGAS answer-quality metrics
Slower — makes additional LLM calls. Set `run_ragas=True` below.

In [ ]:
# pipe = build_pipeline('semantic', use_reranker=True)
# answered = [(q, pipe.answer(q.question)) for q in questions]
# evaluate_experiment(answered, run_ragas=True)['ragas']